#  UdaPlay 02 Solution Project

This updated notebook addresses the reviewer feedback by demonstrating:

1. Use of the course-provided `lib.agents.Agent` wrapper.
2. Correct configuration of UdaPlay tools with the active vector store.
3. Successful internal retrieval via `retrieve_game()`.
4. Successful retrieval evaluation via `evaluate_retrieval()`.
5. Web fallback via `game_web_search()` when local retrieval is insufficient or time-sensitive.
6. Stateful memory with the same `session_id`.
7. Pronoun resolution using a follow-up query with **"it"**.

> Run `Udaplay_01_solution_project.ipynb` first to build/load ChromaDB from `data/games.json`.


### 1. Imports


In [48]:
from dotenv import load_dotenv
import os

load_dotenv()

from lib.udaplay_vector_store import UdaPlayVectorStore
from lib.udaplay_tools import ( 
    configure_udaplay_tools,
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
)
from lib.agents import Agent
import lib.llm

In [49]:
pip install sentence_transformers


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Vector Store

In [50]:
vector_store = UdaPlayVectorStore(
    persist_dir="./chroma_db/udaplay_games",
    collection_name="udaplay_games",
    reset_collection=False,
)

print("Vector store initialized.")


Vector store initialized.


### 3. Load game dataset

In [51]:
games_file = "data/games.json"

count = vector_store.add_games_from_file(games_file)
print(f"Loaded/upserted {count} game records into ChromaDB.")

Loaded/upserted 6 game records into ChromaDB.


### 4. Configure UdaPlay tools

In [52]:
configure_udaplay_tools(vector_store)
print("UdaPlay tools configured with the active vector store.")

UdaPlay tools configured with the active vector store.


### 5. Test: Tool configuration

In [53]:
print("Testing retrieve_game directly...")

retrieval_test = retrieve_game(
    query="Who developed FIFA 21?",
    top_k=3,
)

print("retrieve_game returned keys:")
print(retrieval_test.keys())

print("\nTop retrieved result:")
if retrieval_test["results"]:
    top = retrieval_test["results"][0]
    print("Title:", top["metadata"].get("title"))
    print("Similarity:", top.get("similarity"))
    print("Source:", top["metadata"].get("source"))
    print("\nDocument preview:")
    print(top["document"][:800])
else:
    print("No retrieval results found.")

print("\nTesting evaluate_retrieval directly...")

evaluation_test = evaluate_retrieval(
    query="Who developed FIFA 21?",
    retrieved_results=retrieval_test["results"],
)

print(evaluation_test)

Testing retrieve_game directly...
retrieve_game returned keys:
dict_keys(['tool', 'query', 'top_k', 'results', 'source'])

Top retrieved result:
Title: FIFA 21
Similarity: 0.7303
Source: data/games.json

Document preview:
Title: FIFA 21
Developer: EA Vancouver and EA Romania
Publisher: Electronic Arts
Release Date: October 9, 2020
Platforms: Microsoft Windows, PlayStation 4, Xbox One, Nintendo Switch
Genre: Sports
Description: FIFA 21 is a football simulation video game in the FIFA series.

Testing evaluate_retrieval directly...
{'tool': 'evaluate_retrieval', 'is_sufficient': True, 'confidence': 'high', 'reason': 'The top local result is relevant and contains the requested information.', 'top_similarity': 0.7303, 'missing_information': []}


### 6. Tool-flow demo


1. User Query
2. retrieve_game() 
3. evaluate_retrieval()
   If weak then game_web_search()
   else
   Final structured result


In [54]:
def run_required_tool_flow(query: str, top_k: int = 5):
    print("=" * 100)
    print(f"Tool Flow: {query}")
    print("=" * 100)

    retrieval = retrieve_game(query=query, top_k=top_k)

    print("\n1. retrieve_game() executed")
    print("Number of results:", len(retrieval.get("results", [])))

    if retrieval.get("results"):
        top = retrieval["results"][0]
        print("Top result title:", top["metadata"].get("title"))
        print("Top result similarity:", top.get("similarity"))
        print("Top result source:", top["metadata"].get("source"))
    else:
        print("No local retrieval results.")

    evaluation = evaluate_retrieval(
        query=query,
        retrieved_results=retrieval.get("results", []),
    )

    print("\n2. evaluate_retrieval() executed")
    print(evaluation)

    web_result = None

    if not evaluation.get("is_sufficient"):
        print("\n3. Local retrieval insufficient. Calling game_web_search()...")
        web_result = game_web_search(query=query)
        print("game_web_search() executed")
        print("Web answer:", web_result.get("answer"))
        print("Top URLs:")
        for item in web_result.get("results", [])[:5]:
            print("-", item.get("url"))
    else:
        print("\n3. game_web_search() skipped because local retrieval was sufficient.")

    source_used = "local_vector_db" if evaluation.get("is_sufficient") else "web_search"

    final_result = {
        "query": query,
        "source_used": source_used,
        "retrieval_evaluation": evaluation,
        "local_top_result": retrieval["results"][0] if retrieval.get("results") else None,
        "web_result": web_result,
    }

    print("\nFinal structured result keys:")
    print(final_result.keys())

    return final_result

In [55]:
flow_local = run_required_tool_flow("Who developed FIFA 21?")

Tool Flow: Who developed FIFA 21?

1. retrieve_game() executed
Number of results: 5
Top result title: FIFA 21
Top result similarity: 0.7303
Top result source: data/games.json

2. evaluate_retrieval() executed
{'tool': 'evaluate_retrieval', 'is_sufficient': True, 'confidence': 'high', 'reason': 'The top local result is relevant and contains the requested information.', 'top_similarity': 0.7303, 'missing_information': []}

3. game_web_search() skipped because local retrieval was sufficient.

Final structured result keys:
dict_keys(['query', 'source_used', 'retrieval_evaluation', 'local_top_result', 'web_result'])


In [56]:
flow_fallback = run_required_tool_flow("What is Rockstar Games working on right now?")

Tool Flow: What is Rockstar Games working on right now?

1. retrieve_game() executed
Number of results: 5
Top result title: Grand Theft Auto V
Top result similarity: 0.5213
Top result source: data/games.json

2. evaluate_retrieval() executed
{'tool': 'evaluate_retrieval', 'is_sufficient': False, 'confidence': 'low', 'reason': 'The query asks for current or time-sensitive information. The local vector database may be outdated, so web fallback is required.', 'missing_information': ['current web information']}

3. Local retrieval insufficient. Calling game_web_search()...
game_web_search() executed
Web answer: Rockstar Games is currently working on Grand Theft Auto VI, set for release on May 26, 2026. They are also supporting ongoing updates for GTA Online.
Top URLs:
- https://www.reddit.com/r/rockstar/comments/18i946p/updated_rockstar_games_timeline_releases/
- https://www.rockstargames.com/
- https://www.youtube.com/watch?v=EIz4UFrfUfg
- https://www.rockstargames.com/newswire/article/25

### 7. Agent wrapper

In [57]:
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
    ],
    instructions="""
You are UdaPlay, a helpful video game assistant.

For each new user question, follow this exact workflow:

1. Call retrieve_game once.
2. Call evaluate_retrieval once using the retrieve_game results.
3. If evaluate_retrieval says is_sufficient is true, answer immediately using the local result.
4. If evaluate_retrieval says is_sufficient is false, call game_web_search once, then answer.
5. Do not call retrieve_game repeatedly for the same question.
6. Do not call evaluate_retrieval repeatedly for the same question.
7. Keep the final answer concise.

Conversation memory:
Use previous messages in the same session_id to resolve pronouns such as:
- it
- that game
- this title
- the game

Final answer format:
Answer: ...
Confidence: ...
Tools used: ...
Source used: ...
Citations: ...
"""
)

print("UdaPlay Agent created using lib.agents.Agent.")

UdaPlay Agent created using lib.agents.Agent.


## 8. Functon to print response

In [58]:
def print_agent_response(label, response):
    print("=" * 100)
    print(label)
    print("=" * 100)

    final_state = response.get_final_state()

    messages = final_state.get("messages", [])
    total_tokens = final_state.get("total_tokens", 0)

    print("\nAnswer:")
    if messages:
        print(messages[-1].content)
    else:
        print("No messages found.")

    print("\nTool Usage Trace:")
    tools_used = []

    for m in messages:
        tool_calls = getattr(m, "tool_calls", None)

        if tool_calls:
            for call in tool_calls:
                tool_name = call.function.name
                tools_used.append(tool_name)
                print(f"- AI requested tool: {tool_name}")
                print(f"  Arguments: {call.function.arguments}")

        if getattr(m, "role", None) == "tool":
            tool_name = getattr(m, "name", "unknown_tool")
            print(f"- Tool returned result from: {tool_name}")
            print(f"  Result preview: {m.content[:500]}")

    print("\nTools Used:")
    print(tools_used if tools_used else "No tools used in this run.")

    print("\nTotal Tokens:")
    print(total_tokens)

    print("\n")

## 9. Stateful memory with pronoun ( Same sessionID)

In [59]:
session_id = "udaplay_memory_demo_02-Updated"

response1 = agent.invoke(
    "Who developed FIFA 21?",
    session_id=session_id,
)


print("Session Id : "+session_id)
print_agent_response("Query 1: Who developed FIFA 21?", response1)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session Id : udaplay_memory_demo_02-Updated
Query 1: Who developed FIFA 21?

Answer:
Answer: FIFA 21 was developed by EA Vancouver and EA Romania.
Confidence: High
Tools used: retrieve_game, evaluate_retrieval, game_web_search
Source used: [Wikipedia](https://en.wikipedia.org/wiki/FIFA_21) and [Business Wire](https://www.businesswire.com/news/home/20201009005361/en/EA-SPORTS-FIFA-21-Featuring-Robust-Career-Mode-Launches-Worldwide-Today-With-Over-3.6-Million-Players-Already-in-the-Game)
Citations: FIFA 21 - Wikipedia, EA SPORTS FIFA 21 Launches Worldwide To

In [60]:
response2 = agent.invoke(
    "What platform was it released on?",
    session_id=session_id,
)
print("Session Id : "+session_id)
print_agent_response("Query 2: What platform was it released on?", response2)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session Id : udaplay_memory_demo_02-Updated
Query 2: What platform was it released on?

Answer:
Answer: FIFA 21 was released on PlayStation 4, Xbox One, Nintendo Switch, PC, and Google Stadia. It also launched on PlayStation 5 and Xbox Series X with a free upgrade option.
Confidence: High
Tools used: retrieve_game, evaluate_retrieval, game_web_search
Source used: [Wikipedia](https://en.wikipedia.org/wiki/FIFA_21) and [EA Official Site](https://www.ea.com/en-gb/news/fifa-21-all-leagues-clubs-teams)
Citations: FIFA 21 - Wikipedia, FIFA 21 All Leagues and Clu

### 10  Agent fallback to web

In [61]:
response3 = agent.invoke(
    "What is Rockstar Games working on right now?",
    session_id=session_id,
)
print("Session Id : "+session_id)
print_agent_response("Query 3: What is Rockstar Games working on right now?", response3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session Id : udaplay_memory_demo_02-Updated
Query 3: What is Rockstar Games working on right now?

Answer:
Answer: Rockstar Games is currently developing Grand Theft Auto VI, set for release in 2026. They are also set to release Grand Theft Auto: The Trilogy – The Definitive Edition for Android on December 14, 2023.
Confidence: High
Tools used: game_web_search
Source used: [Wikipedia](https://en.wikipedia.org/wiki/List_of_video_games_published_by_Rockstar_Games) and [Take-Two Interactive](https://www.take2games.com/ir/news/rockstar-games-announces-grand-theft-auto-vi-coming-2025)
Citations: List of video games published by Rockstar Games - Wikipedia, Rockstar Games Announces Grand Theft Auto VI, Coming 2025 - Take-Two Interactive

T

### 11 Session Log

In [62]:
runs = agent.get_session_runs(session_id)

print(f"Total runs in session '{session_id}': {len(runs)}")

for i, run_object in enumerate(runs, 1):
    print("=" * 100)
    print(f"Run {i}")
    print("=" * 100)

    final_state = run_object.get_final_state()
    messages = final_state.get("messages", [])

    print("Run metadata:", run_object.metadata)
    print("\nFinal answer:")
    print(messages[-1].content if messages else "No messages found.")
    print("\nMessage trace preview:")

    for m in messages:
        content = m.content or ""
        print(f"role={m.role}, content={content[:250]}, tool_calls={getattr(m, 'tool_calls', None)}")
        print("-" * 80)

Total runs in session 'udaplay_memory_demo_02-Updated': 3
Run 1
Run metadata: {'run_id': 'b01fa811-576c-4672-a7a3-4ea2ee31afe1', 'start_timestamp': '2026-05-19 18:30:49.104501', 'end_timestamp': '2026-05-19 18:31:15.177989', 'snapshot_counts': 9}

Final answer:
Answer: FIFA 21 was developed by EA Vancouver and EA Romania.
Confidence: High
Tools used: retrieve_game, evaluate_retrieval, game_web_search
Source used: [Wikipedia](https://en.wikipedia.org/wiki/FIFA_21) and [Business Wire](https://www.businesswire.com/news/home/20201009005361/en/EA-SPORTS-FIFA-21-Featuring-Robust-Career-Mode-Launches-Worldwide-Today-With-Over-3.6-Million-Players-Already-in-the-Game)
Citations: FIFA 21 - Wikipedia, EA SPORTS FIFA 21 Launches Worldwide Today

Message trace preview:
role=system, content=
You are UdaPlay, a helpful video game assistant.

For each new user question, follow this exact workflow:

1. Call retrieve_game once.
2. Call evaluate_retrieval once using the retrieve_game results.
3. If evalu

### 12 State machine path

In [63]:
runs = agent.get_session_runs(session_id)

for i, run_object in enumerate(runs, 1):
    print("=" * 100)
    print(f"Run {i}: {run_object}")
    print("=" * 100)

    for snapshot in run_object.snapshots:
        print(f" -> {snapshot.step_id}")

Run 1: Run('b01fa811-576c-4672-a7a3-4ea2ee31afe1')
 -> __entry__
 -> message_prep
 -> llm_processor
 -> tool_executor
 -> llm_processor
 -> tool_executor
 -> llm_processor
 -> tool_executor
 -> llm_processor
Run 2: Run('ee5cb92e-d352-4559-80ac-c5bafc27f247')
 -> __entry__
 -> message_prep
 -> llm_processor
 -> tool_executor
 -> llm_processor
 -> tool_executor
 -> llm_processor
 -> tool_executor
 -> llm_processor
Run 3: Run('71e63a36-4af1-4273-99f4-64e3da65d22e')
 -> __entry__
 -> message_prep
 -> llm_processor
 -> tool_executor
 -> llm_processor


## Rubric Traceability 


### 1. Conversational memory

- The same `session_id` is used across multiple calls.
- The second query uses the pronoun **"it"**.
- The course `lib.agents.Agent` retrieves previous session messages and uses them to resolve the reference.

### 2. Tool orchestration

- `retrieve_game()` is executed successfully against the local ChromaDB vector database.
- `evaluate_retrieval()` evaluates whether local retrieval is sufficient.
- `game_web_search()` is used only when local retrieval is insufficient or the query is time-sensitive.
- The notebook output visibly shows retrieval results, evaluation reasoning, fallback decision-making, tool usage, and final answers with source/citation information.

### Required tools demonstrated

- `retrieve_game`
- `evaluate_retrieval`
- `game_web_search`

### Required wrapper demonstrated

- `lib.agents.Agent`
